In [1]:
import os

os.listdir("/content")

['.config', 'Rd.zip', 'sample_data']

In [3]:
import zipfile

zip_path = "/content/Rd.zip"

extract_path = "/content/Rd"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully")

Dataset Extracted Successfully


In [7]:
import os

dataset_path = "/content/Rd/DATASET"

print(os.listdir(dataset_path)[:10])

['train', 'test']


In [8]:
import os

train_path = "/content/Rd/DATASET/train"

print(os.listdir(train_path))

['7', '3', '1', '4', '5', '2', '6']


In [10]:
# ============================================
# IMPORTS
# ============================================

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import *
from tensorflow.keras.optimizers import Adam

# ============================================
# SETTINGS
# ============================================

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 35

dataset_path = "/content/Rd/DATASET/train"

# ============================================
# DATASET
# ============================================

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# ============================================
# CLASS NAMES
# ============================================

print(train_data.class_indices)

# ============================================
# MODEL
# ============================================

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(7, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# ============================================
# COMPILE
# ============================================

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ============================================
# CALLBACKS
# ============================================

checkpoint = ModelCheckpoint(
    "best_emotion_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

earlystop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2
)

callbacks = [checkpoint, earlystop, reduce_lr]

# ============================================
# TRAIN STAGE 1
# ============================================

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks
)

# ============================================
# FINE TUNING
# ============================================

base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

# ============================================
# SAVE MODEL
# ============================================

model.save("final_emotion_model.h5")

print("MODEL TRAINED SUCCESSFULLY")

Found 9819 images belonging to 7 classes.
Found 2452 images belonging to 7 classes.
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 0.3012 - loss: 2.1068

307/307 ━━━━━━━━━━━━━━━━━━━━ 227s 638ms/step - accuracy: 0.3574 - loss: 1.8874 - val_accuracy: 0.4927 - val_loss: 1.4336 - learning_rate: 1.0000e-04
Epoch 2/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - accuracy: 0.4233 - loss: 1.6074

307/307 ━━━━━━━━━━━━━━━━━━━━ 161s 525ms/step - accuracy: 0.4342 - loss: 1.5787 - val_accuracy: 0.5338 - val_loss: 1.3105 - learning_rate: 1.0000e-04
Epoch 3/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - accuracy: 0.4561 - loss: 1.4826

307/307 ━━━━━━━━━━━━━━━━━━━━ 159s 518ms/step - accuracy: 0.4619 - loss: 1.4644 - val_accuracy: 0.5355 - val_loss: 1.2683 - learning_rate: 1.0000e-04
Epoch 4/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.4778 - loss: 1.4234

307/307 ━━━━━━━━━━━━━━━━━━━━ 160s 522ms/step - accuracy: 0.4818 - loss: 1.4168 - val_accuracy: 0.5591 - val_loss: 1.2414 - learning_rate: 1.0000e-04
Epoch 5/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step - accuracy: 0.4940 - loss: 1.3763

...

307/307 ━━━━━━━━━━━━━━━━━━━━ 167s 543ms/step - accuracy: 0.5694 - loss: 1.1791 - val_accuracy: 0.6170 - val_loss: 1.0953 - learning_rate: 1.0000e-04
Epoch 13/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.5706 - loss: 1.1553

307/307 ━━━━━━━━━━━━━━━━━━━━ 157s 511ms/step - accuracy: 0.5723 - loss: 1.1590 - val_accuracy: 0.6195 - val_loss: 1.0762 - learning_rate: 1.0000e-04
Epoch 14/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.5738 - loss: 1.1342

307/307 ━━━━━━━━━━━━━━━━━━━━ 157s 511ms/step - accuracy: 0.5763 - loss: 1.1460 - val_accuracy: 0.6272 - val_loss: 1.0760 - learning_rate: 1.0000e-04
Epoch 15/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 0.5828 - loss: 1.1374

307/307 ━━━━━━━━━━━━━━━━━━━━ 162s 528ms/step - accuracy: 0.5886 - loss: 1.1306 - val_accuracy: 0.6285 - val_loss: 1.0618 - learning_rate: 1.0000e-04
Epoch 1/35
307/307 ━━━━━━━━━━━━━━━━━━━━ 224s 621ms/step - accuracy: 0.4955 - loss: 1.3855 - val_accuracy: 0.5852 - val_loss: 1.1951 - learning_rate: 1.0000e-05
Epoch 2/35
307/307 ━━━━━━━━━━━━━━━━━━━━ 164s 535ms/step - accuracy: 0.5256 - loss: 1.3002 - val_accuracy: 0.5877 - val_loss: 1.1648 - learning_rate: 1.0000e-05
Epoch 3/35
307/307 ━━━━━━━━━━━━━━━━━━━━ 161s 525ms/step - accuracy: 0.5358 - loss: 1.2653 - val_accuracy: 0.5983 - val_loss: 1.1510 - learning_rate: 2.0000e-06
Epoch 4/35
307/307 ━━━━━━━━━━━━━━━━━━━━ 161s 524ms/step - accuracy: 0.5434 - loss: 1.2544 - val_accuracy: 0.5889 - val_loss: 1.1550 - learning_rate: 2.0000e-06
Epoch 5/35
307/307 ━━━━━━━━━━━━━━━━━━━━ 164s 535ms/step - accuracy: 0.5426 - loss: 1.2497 - val_accuracy: 0.5995 - val_loss: 1.1423 - learning_rate: 4.0000e-07


MODEL TRAINED SUCCESSFULLY


In [11]:
from google.colab import files

files.download("final_emotion_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
loss, acc = model.evaluate(val_data)

print("Validation Accuracy:", acc * 100)

77/77 ━━━━━━━━━━━━━━━━━━━━ 34s 439ms/step - accuracy: 0.5681 - loss: 1.2134
Validation Accuracy: 56.81076645851135


In [13]:
from tensorflow.keras.models import load_model

model = load_model("final_emotion_model.h5")

In [14]:
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,

    validation_split=0.2,

    rotation_range=35,
    zoom_range=0.3,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,

    horizontal_flip=True,

    brightness_range=[0.7,1.3]
)

In [15]:
for layer in model.layers:
    layer.trainable = True

In [16]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.000001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=50
)

Epoch 1/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 312s 759ms/step - accuracy: 0.3990 - loss: 1.6420 - val_accuracy: 0.4841 - val_loss: 1.4088
Epoch 2/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 174s 568ms/step - accuracy: 0.4038 - loss: 1.6279 - val_accuracy: 0.4502 - val_loss: 1.4803
Epoch 3/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 173s 562ms/step - accuracy: 0.4202 - loss: 1.5980 - val_accuracy: 0.4657 - val_loss: 1.4354
Epoch 4/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 173s 565ms/step - accuracy: 0.4219 - loss: 1.5759 - val_accuracy: 0.4694 - val_loss: 1.4338
Epoch 5/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 173s 564ms/step - accuracy: 0.4246 - loss: 1.5716 - val_accuracy: 0.4743 - val_loss: 1.4333
Epoch 6/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 171s 556ms/step - accuracy: 0.4226 - loss: 1.5504 - val_accuracy: 0.4788 - val_loss: 1.4150
Epoch 7/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 172s 561ms/step - accuracy: 0.4331 - loss: 1.5372 - val_accuracy: 0.4951 - val_loss: 1.3877
Epoch 8/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 170s 555ms/step - accuracy: 0.4374 -